# 02 - Limpieza y transformacion de combinaciones de componentes

Este notebook esta reservado para preparar los datos del enfoque de combinaciones de componentes.

Tareas sugeridas:
- normalizar textos de `Composition`;
- separar componentes multiples;
- generar una estructura apta para analisis por combinacion.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.load_data import load_medicine_data
from src.enfoque_01_combinaciones_componentes.cleaning import run_cleaning_pipeline

# 1. Cargar datos raw SIN intentar descargar
df_raw = load_medicine_data(download_if_missing=False)
print(f"Shape raw: {df_raw.shape}")

# 2. Ejecutar limpieza completa
df_clean = run_cleaning_pipeline(df_raw, save=True)

# 3. Verificar resultado
df_clean[["Medicine Name", "Composition", "components_list",
          "n_components", "size_category"]].head(10)

Shape raw: (11825, 9)
INICIO DEL PIPELINE DE LIMPIEZA
[eliminar_duplicados] Files antes: 11825 | Eliminados: 84 | Files despues: 11741
[normalizar_composicion] Normalización de espacios completada.
[añadir_columnas_componentes] Distribución por tamaño de combinación:
    mono         ->  7019 medicamentos
    duo          ->  3569 medicamentos
    trio         ->   929 medicamentos
    cuádruple    ->   147 medicamentos
    complejo     ->    77 medicamentos
[flag_anomalies] Registros con anomalía en Composition: 0
-------------------------------------------------------
Forma final del DataFrame: (11741, 13)
[run_cleaning_pipeline] CSV limpio guardado en: /home/lucasmoncada/SCY1101-Exp1-Drug-Data-Analysis/data/processed/medicine_cleaned.csv


,Medicine Name,Composition,components_list,n_components,size_category
0,Avastin 400mg Injection,Bevacizumab (400mg),[Bevacizumab],1,mono
1,Augmentin 625 Duo Tablet,Amoxycillin (500mg) + Clavulanic Acid (125mg),"[Amoxycillin, Clavulanic Acid]",2,duo
2,Azithral 500 Tablet,Azithromycin (500mg),[Azithromycin],1,mono
3,Ascoril LS Syrup,Ambroxol (30mg/5ml) + Levosalbutamol (1mg/5ml)...,"[Ambroxol, Levosalbutamol, Guaifenesin]",3,trio
4,Aciloc 150 Tablet,Ranitidine (150mg),[Ranitidine],1,mono
5,Allegra 120mg Tablet,Fexofenadine (120mg),[Fexofenadine],1,mono
6,Avil 25 Tablet,Pheniramine (25mg),[Pheniramine],1,mono
7,Aricep 5 Tablet,Donepezil (5mg),[Donepezil],1,mono
8,Amoxyclav 625 Tablet,Amoxycillin (500mg) + Clavulanic Acid (125mg),"[Amoxycillin, Clavulanic Acid]",2,duo
9,Atarax 25mg Tablet,Hydroxyzine (25mg),[Hydroxyzine],1,mono


In [2]:
import importlib
import src.enfoque_01_combinaciones_componentes.transform as t
importlib.reload(t)
from src.enfoque_01_combinaciones_componentes.transform import run_transform_pipeline

# df_clean ya lo tienes de la celda anterior
df_exploded, df_pairs, cooc_matrix = run_transform_pipeline(df_clean, top_n=20, save=True)

# Verificar cada resultado
print(df_exploded[["Medicine Name", "component", "n_components"]].head(5))
print(df_pairs[["Medicine Name", "comp_a", "comp_b"]].head(5))
print(cooc_matrix.iloc[:5, :5])

INICIO DEL PIPELINE DE TRANSFORMACIONES
[explotar_componentes] Filas antes: 11741 -> después: 17957
[explotar_componentes] Componentes únicos: 1058


[explotar_pares] Total de pares generados: 8227
[explotar_pares] Pares únicos: 1455
[construir_matriz_coocurrencia] Matriz generada 20×20
-------------------------------------------------------
df_exploded shape : (17957, 14)
df_pairs shape    : (8227, 15)
cooc_matrix shape : (20, 20)
[run_transform_pipeline] Archivos guardados en: /home/lucasmoncada/SCY1101-Exp1-Drug-Data-Analysis/data/processed
              Medicine Name        component  n_components
0   Avastin 400mg Injection      Bevacizumab             1
1  Augmentin 625 Duo Tablet      Amoxycillin             2
2  Augmentin 625 Duo Tablet  Clavulanic Acid             2
3       Azithral 500 Tablet     Azithromycin             1
4          Ascoril LS Syrup         Ambroxol             3
              Medicine Name       comp_a           comp_b
0  Augmentin 625 Duo Tablet  Amoxycillin  Clavulanic Acid
1          Ascoril LS Syrup     Ambroxol      Guaifenesin
2          Ascoril LS Syrup     Ambroxol   Levosalbutamol
3          Asc

In [3]:
from src.enfoque_01_combinaciones_componentes.validation import run_validation_pipeline

reporte = run_validation_pipeline(df_raw, df_clean)

# Guardar el MD5 para futuras verificaciones
print(f"\nMD5 del CSV original: {reporte['checksum']['md5']}")

INICIO DEL PIPELINE DE VALIDACIÓN

[1/4] Verificando checksum MD5...
[verificar_checksum] Medicine_Details.csv
    MD5     : 93656e73e4e8fbf7dd8da9ecfab7ce07
    Tamaño  : 4,360,239 bytes
    Estado  : SIN_REFERENCIA

[2/4] Validando esquema del DataFrame raw...
[validar_esquema] Estado: VÁLIDO

[3/4] Comparando shapes antes/después de limpieza...
[comparar_shapes]
    Filas raw           : 11,825
    Filas limpias       : 11,741
    Filas eliminadas    : 84 (0.71%)
    Columnas raw        : 9
    Columnas clean      : 13
    Columnas nuevas     : ['components_list', 'composition_anomaly', 'n_components', 'size_category']

[4/4] Validando columnas del DataFrame limpio...
[validar_columnas_clean] Estado: VÁLIDO

-------------------------------------------------------
[exportar_reporte] Reporte guardado en: /home/lucasmoncada/SCY1101-Exp1-Drug-Data-Analysis/outputs/reports/reporte_integridad.json
VALIDACIÓN COMPLETA

MD5 del CSV original: 93656e73e4e8fbf7dd8da9ecfab7ce07
